# Finding vulnerabilities in the masked Decompose fonction of Dilithium:

The purpose of this notebook is to generate the datasets needed for the ANOVA and to create the template, using the ChipWhisperer.

In [1]:
!pwd

/media/sf_vm_shared/Dev/ML-DSA/Security/Decompose/artifacts/artifacts/clean_template


In [2]:
# Loading auxiliary functions 
probable_path_to_helpers_functions = !find ../Common_functions -name "Helpers.py"

# If the Helper.py file is not found and you don't need it, comment this cell
# If the Helper.py file is not found and you need it, something went wrong ...
print(f"Probable path to Helpers functions:")
print(f">>> {probable_path_to_helpers_functions}")
probable_path_to_helpers_functions = probable_path_to_helpers_functions[0]

%run -i $probable_path_to_helpers_functions

Probable path to Helpers functions:
>>> ['../Common_functions/Helpers.py']


## Preliminaries
---

- For ML-DSA 2 choose  `K = 2`
- For ML-DSA 3 choose  `K = 3`
- For ML-DSA 5 choose  `K = 5`

In [5]:
MODE = 2
# K = 3
# K = 5

In [6]:
# Loading ml-dsa parameters according to the chosen security level K
%run -i ../Common_functions/MLDSA_parameters.py {MODE}

# Loading auxiliary functions
%run -i ../Common_functions/MLDSA_functions.py

# Loading auxiliary functions
%run -i ../Common_functions/Additional_functions.py

In [7]:
# Loading auxiliary functions for ChipWhisperer communication
%run -i ../Chipwhisperer_functions.py

In [8]:
# Importing useful libraries
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from tqdm.notebook import trange
import copy
import random 

import scared
import estraces

from collections import Counter

from scalib.preprocessing import Quantizer
from scalib.metrics import Ttest, SNR

import struct 

from scipy.signal import correlate
from scipy.ndimage import shift

In [9]:
# Setting default size of figures in Matplotlib
plt.rcParams["figure.figsize"] = (13,3) 

# Set the maximum display width for NumPy arrays
np.set_printoptions(linewidth=sys.maxsize)

# Adjusting the display of tables
np.set_printoptions(threshold=sys.maxsize)

In [10]:
# Matplotlib constants 
span_color = "#FFC069"
color0 = "darkblue"
color1 = "coral"

### Setup of the ChipWhisperer

In [11]:
# Base Scope for ChipWhisperer Lite 
SCOPETYPE = 'OPENADC'

# ChipWhisperer Lite used Cortex-M4 
PLATFORM = 'CWLITEARM'

# Project Targeted
CRYPTO_TARGET ='DECOMPOSE2'

# SimpleSerial version used
SS_VER = 'SS_VER_1_1'

In [12]:
# Detecting where is the simpleserial find, if any
probable_path_to_chipwhisperer_setup_notebook = !find /home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/jupyter/Setup_Scripts/ -name "Setup_Generic.ipynb"

print(f"Probable path to ChipWhisperer jupyter setup script:")
print(f">>> {probable_path_to_chipwhisperer_setup_notebook}")

Probable path to ChipWhisperer jupyter setup script:
>>> ['/home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb']


In [13]:
# # Change the path to match the location of your file Setup_Generic.ipynb
# path_to_chipwhisperer_setup_notebook = ...
path_to_chipwhisperer_setup_notebook = probable_path_to_chipwhisperer_setup_notebook[0]

In [14]:
%run $path_to_chipwhisperer_setup_notebook

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.64.0) is outdated - latest is 0.65.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 19527225                  to 82904241                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538459                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538459.0        

In [15]:
# Number of samples that the ChipWhisperer has to record per trace

# Maximum number of samples allowed in [0, 24400]
scope.adc.samples = 4400

### Compiling the code

In [16]:
# Detecting where is the simpleserial location, if any
probable_path_to_simpleserial = !find ~ -name simpleserial-{CRYPTO_TARGET.lower()}

print(f"Probable path to communication protocol with the {CRYPTO_TARGET} code: ")
print(f">>> {probable_path_to_simpleserial}")

Probable path to communication protocol with the DECOMPOSE2 code: 
>>> ['/home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/firmware/mcu/simpleserial-decompose2']


In [17]:
# # Change the path to match the location of the simpleserial code 
# path_to_simpleserial = ... 
path_to_simpleserial = probable_path_to_simpleserial[-1]

In [18]:
%%bash -s "$path_to_simpleserial" "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER" 
cd $1
make -s clean PLATFORM=$2 CRYPTO_TARGET=$3 SS_VER=$4 
make PLATFORM=$2 CRYPTO_TARGET=$3 SS_VER=$4 

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:12.2.rel1-1) 12.2.1 20221205
Copyright (C) 2022 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Compiling:
-en     simpleserial-decompose2.c ...


simpleserial-decompose2.c: In function 'setup2':
simpleserial-decompose2.c:473:41: warning: passing argument 3 of 'simpleserial_put' from incompatible pointer type [-Wincompatible-pointer-types]
  473 |     simpleserial_put('r', 4*N_SHARES*2, &rz2_rz3);
      |                                         ^~~~~~~~
      |                                         |
      |                                         uint32_t (*)[4] {aka long unsigned int (*)[4]}
In file included from simpleserial-decompose2.c:5:
../simpleserial/simpleserial.h:63:54: note: expected 'uint8_t *' {aka 'unsigned char *'} but argument is of type 'uint32_t (*)[4]' {aka 'long unsigned int (*)[4]'}
   63 | void simpleserial_put(char c, uint8_t size, uint8_t* output);
      |                                             ~~~~~~~~~^~~~~~


-e Done!
.
Compiling:
-en     ../simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../hal/hal.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_hal.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_hal_lowlevel.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_sysmem.c ...
-e Done!
.
Assembling: ../hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I../simpleserial/ -I../hal/ -I../hal/ -I../hal//stm32f3 -I../hal//stm32f3/CMSIS -I../hal//stm32f3/CMSIS/core -I../hal//stm32f3/CMSIS/device -I../hal//stm32f4/Legacy -I../simpleserial/ -I../crypto/ ../hal//stm32f3/stm32f3_startup.S -o objdir-CWLITEARM/stm32f3_startup.o
.
LINKING:
-en     simpleserial-decompose2-CWLITEARM.elf ...
Memory region         Used Size  Region Size  %age Used
             RAM:        1704 B        4

### Loading the executable into the ChipWhisperer

In [19]:
# Detecting where is the simpleserial find, if any
probable_path_to_executable = !find ~/ -name simpleserial-{CRYPTO_TARGET.lower()}-{PLATFORM}.hex

print(f"Probable path to communication protocol with the {CRYPTO_TARGET} code: ")
print(f">>> {probable_path_to_executable}")

Probable path to communication protocol with the DECOMPOSE2 code: 
>>> ['/home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/firmware/mcu/simpleserial-decompose2/simpleserial-decompose2-CWLITEARM.hex']


In [20]:
# # Change the path to match the executable code 
# path_to_executable = ...
path_to_executable = probable_path_to_executable[-1]

In [21]:
cw.program_target(scope, prog, path_to_executable)

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 8567 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 8567 bytes


-----------
## Communication with the board

In [22]:
vhex = np.vectorize(hex)

def cw_to_int(hexstr):
    beh = bytes.fromhex(hexstr)
    leh = struct.unpack("<I", beh)[0]
    return leh

def cw_to_int2(byte_list):
    leb = bytes(reversed(byte_list))
    num = int.from_bytes(leb, "big", signed=True)
    return num

def int_to_cw2(num):
    return [int(byy) for byy in reversed((num).to_bytes(4, "big", signed=True))]

def int_to_cw(num, b_ = 4):
    leb = num.to_bytes(b_, "big", signed=True)
    beh = "".join(format(byte, "02x") for byte in reversed(leb))
    return beh

def from_intarray_to_kybermsg(arrray):
    return "".join([f"{byyte:08x}"[::-1] for byyte in arrray])

def HW(n):
    return bin(n).count('1')

def compressed(val):
    msg_int = cw_to_int2(val)
    if msg_int > -Q//4:
        return [byy for byy in reversed((0).to_bytes(4, "big", signed=True))]
    return [byy for byy in reversed((1).to_bytes(4, "big", signed=True))]

def bytes_to_hex_str(bytess):
    return ''.join('{:02x}'.format(x) for x in bytess)

def int_array_to_hex_str(int_array):
    return "".join([int_to_cw(int(int_array[i])) for i in range(len(int_array))])

def hex_str_to_int_array(hex_str, hex_str_len = 8):
    int_aray = []
    for offset in range(0, len(hex_str), hex_str_len):
        int_aray.append(cw_to_int(hex_str[offset:offset + hex_str_len]))
    return int_aray

def hw(string):
    return string.count("1")
v__hw = np.vectorize(hw)

In [23]:
def trace_worker(scope, target, n_samples, w_shares):
    t = np.zeros((n_samples))
    scope.arm()
    target.simpleserial_write('d', bytearray(w_shares))
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
    else:
        t = scope.get_last_trace()[:n_samples]
    test_output = target.simpleserial_read("r", 2*2*4)
    z2_share0, z2_share1 =  cw_to_int2(test_output[:4]), cw_to_int2(test_output[4:8])
    z3_share0, z3_share1 =  cw_to_int2(test_output[8:12]), cw_to_int2(test_output[12:16])

    z2, z3 = z2_share0 ^ z2_share1, z3_share0 ^ z3_share1
    return t, z2, z3

def random_arithmetic_shares(w_coeff):
    n_shares = 2
    tmp = np.zeros(n_shares, dtype=np.uint32)
    for i in range(n_shares - 1):
        tmp[i] = (np.random.randint(np.iinfo(np.uint32).max, dtype=np.uint32)) % Q
        diff = np.int64(w_coeff) - np.int64(tmp[i])
        tmp[n_shares - 1] = diff % Q
    return tmp

# Capture traces of b2a conversion of 'n_w_coeff' random boolean share 
# pairs masking y-coefficients in the range [w_intermediate-w_range, w_intermediate+w_range[. 
# Values < w_intermediate are labeled 0, values >= w_intermediate are labeled 1.
def capture_profiling_traces(scope, w_intermediate=GAMMA2, w_range=16, n_w_coeff=2000):
    target = cw.target(scope, cw.targets.SimpleSerial, flush_on_err=False)
    n_samples = 20000
    scope.adc.offset = 0
    scope.clock.adc_src = 'clkgen_x1'
    scope.adc.samples = n_samples

    # Number of coefficietns
    n_traces = (w_range*2) * n_w_coeff
    labels = np.zeros(n_traces, dtype=np.uint16)
    traces = np.zeros((n_traces, n_samples))
    boolean_shares = np.zeros(2, dtype=np.uint32)

    label = 0
    coeff_idx = 0
    for w_coeff in trange(w_intermediate - w_range, (w_intermediate + w_range), desc = "Iterating on w"):
        if w_coeff == w_intermediate+1:
            label = 1
        idx = coeff_idx * n_w_coeff
        for i in (range(n_w_coeff)):
            w_shares = random_arithmetic_shares(w_coeff)                
            labels[idx + i] = label
            traces[idx + i] = trace_worker(scope, target, n_samples, w_shares)
        coeff_idx += 1

    return traces, labels


In [24]:
def align_traces(data, window_center, window_size, max_shift):
    """
    Aligns traces in a large array based on cross-correlation within a specific window.
    
    Parameters:
    - data: np.array of shape (N_traces, N_samples)
    - window_center: int, index of the feature to align on (approximate)
    - window_size: int, width of the window to use for correlation
    - max_shift: int, maximum expected shift (to limit search)
    
    Returns:
    - aligned_data: np.array, the realigned traces
    - shifts: np.array, the integer shifts applied to each trace
    """
    n_traces, n_samples = data.shape
    
    # 1. Define Reference (using the mean can be more robust than using just data[0])
    # However, for speed on large data, we can just use the first trace if it's clean.
    ref_trace = np.median(data, axis=0) # Median is robust to outliers
    
    # Define the Region of Interest (ROI) for the reference
    start = max(0, window_center - window_size // 2)
    end = min(n_samples, window_center + window_size // 2)
    ref_window = ref_trace[start:end]
    
    # Pre-allocate output
    aligned_data = np.zeros_like(data)
    shifts = np.zeros(n_traces, dtype=int)
    
    # 2. Iterate and Calculate Lags
    # Note: We loop because vectorizing cross-corr on different lags is tricky.
    # Given the window is small, this loop will be fast enough.
    
    for i in range(n_traces):
        # Extract the same window from the current trace
        # We widen the search window by 'max_shift' to ensure we catch the feature
        search_start = max(0, start - max_shift)
        search_end = min(n_samples, end + max_shift)
        
        curr_window = data[i, search_start:search_end]
        
        # Cross-correlate
        # 'valid' mode means we slide the smaller ref_window over the larger curr_window
        xcorr = correlate(curr_window, ref_window, mode='valid')
        
        # Find the index of maximum correlation
        # The peak in xcorr corresponds to the best alignment position
        best_idx = np.argmax(xcorr)
        
        # Calculate the shift relative to the center
        # The logic depends on 'valid' correlation output size
        # Expected peak index if aligned is 'max_shift'
        shift_amount = best_idx - max_shift
        
        shifts[i] = shift_amount
        
        # 3. Apply Shift
        # Integers shifts are faster using np.roll, but it wraps around. 
        # For valid signal data, we usually want to fill with 0s or NaNs.
        # Custom roll with padding:
        if shift_amount == 0:
            aligned_data[i] = data[i]
        elif shift_amount > 0: # Trace is delayed, shift left (negative roll) or right?
            # If shift is positive, it means the trace appears LATER than ref.
            # We need to shift it LEFT (negative index) to align.
            # However, let's stick to standard convention: shift value is how much to move trace.
            
            # Using simple roll (circular buffer) - fast
            aligned_data[i] = np.roll(data[i], -shift_amount)
            # Zero out the wrapped part if necessary
            aligned_data[i, -shift_amount:] = 0
            
        else: # shift_amount < 0
            aligned_data[i] = np.roll(data[i], -shift_amount)
            aligned_data[i, :-shift_amount] = 0

    return aligned_data, shifts

# --- Usage Example ---

# 1. Setup Dummy Data (64k traces, 20k samples)
# NOTE: This generates random data, usually you load your real .npy file
# For testing, we use a smaller size, but the logic holds.
# N_TRACES = 64000
# N_SAMPLES = 20000
# data = np.load("your_data.npy") 

# 2. Define Parameters
# You need to know roughly where your signal is.
# Example: If you have a pulse at index 5000
offset_location = 5000 
window_width = 400       # Look at 400 samples around the pulse
max_expected_shift = 50  # We expect shifts no larger than +/- 50 samples

In [230]:
#We retrieve the complete list so we can choose the template values carefully.

def pick_1000_rdm_elmt(L):
    if len(L) < 1000:
        return (1000//len(L))*list(L)
    
    selection = random.sample(L, 1000)
    return selection

Counter({np.int64(11): 1352065, np.int64(12): 1352000, np.int64(10): 1144066, np.int64(13): 1143779, np.int64(9): 817190, np.int64(14): 816475, np.int64(8): 490314, np.int64(15): 489027, np.int64(7): 245157, np.int64(16): 243441, np.int64(6): 100947, np.int64(17): 99231, np.int64(5): 33649, np.int64(18): 32362, np.int64(4): 8855, np.int64(19): 8140, np.int64(3): 1771, np.int64(20): 1485, np.int64(2): 253, np.int64(21): 175, np.int64(1): 23, np.int64(22): 10, np.int64(0): 1})
0
taille: 1000
1
taille: 1989
2
taille: 2748
3
taille: 3748
4
taille: 4748
5
taille: 5748
6
taille: 6748
7
taille: 7748
8
taille: 8748
9
taille: 9748
10
taille: 10748
11
taille: 11748
12
taille: 12748
13
taille: 13748
14
taille: 14748
15
taille: 15748
16
taille: 16748
17
taille: 17748
18
taille: 18748
19
taille: 19748
20
taille: 20748
21
taille: 21623
22
taille: 22623


In [231]:
Y_z2 = np.load("./dump_z2.npy").tolist()
Y_z3 = np.load("./dump_z3.npy").tolist()

Y_max = []
for _ in range(Q-1):
    Y_max.append(max(Y_z2[_],Y_z3[_]))

del Y_z2, Y_z3

Y_max = np.array(Y_max)
print(Counter(Y_max))

template_list=[]
for hw in range(23):
    potential_w = list(np.where(Y_max == hw)[0])
    template_list+=pick_1000_rdm_elmt(potential_w)
    
random.shuffle(template_list)

22623

In [232]:
NB_TRACES = len(template_list)

ets_file_test1 = scared.traces.ETSWriter(f'dataset__dcp2__rdm.ets', overwrite=True)

n_samples = 20000
scope.adc.samples = n_samples
scope.adc.offset = 0
scope.clock.adc_src = 'clkgen_x1'
target = cw.target(scope, cw.targets.SimpleSerial, flush_on_err=False)

# Prepare the ets writer                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
for ind in trange(NB_TRACES, desc = 'Capturing traces for fixed value'):
    w_intermediate1 = int(template_list[ind])
    w_shares1 = random_arithmetic_shares(w_intermediate1)                
    trace1, z2, z3 = trace_worker(scope, target, n_samples, w_shares1)
    ets_file_test1.write_samples(np.array(trace1)[:])
    ets_file_test1.write_meta(tag=f'w', metadata = np.array(int_to_cw2(w_intermediate1), dtype = np.uint8))
    ets_file_test1.write_meta(tag=f'w_share0', metadata = np.array(int_to_cw2(int(w_shares1[0])), dtype = np.uint8))
    ets_file_test1.write_meta(tag=f'w_share1', metadata = np.array(int_to_cw2(int(w_shares1[1])), dtype = np.uint8))
    ets_file_test1.write_meta(tag=f'w_z2', metadata = np.array(int_to_cw2(z2), dtype = np.uint8))
    ets_file_test1.write_meta(tag=f'w_z3', metadata = np.array(int_to_cw2(z3), dtype = np.uint8))

ets_file_test1.close()

Capturing traces for fixed value:   0%|          | 0/22623 [00:00<?, ?it/s]

/home/paco/.local/share/virtualenvs/Formation_ChipWhisperer-_97ku2Dw/lib/python3.11/site-packages/estraces/formats/ets_writer.py:221: DeprecationWarning: This method is deprecated and will be removed in a future version. Use write_samples instead.
  warnings.warn('This method is deprecated and will be removed in a future version. Use write_samples instead.', DeprecationWarning)


In [233]:
disconnect_cw()